# Tuần 2 · So sánh hai cách khởi tạo LSTM
Phiên bản 2.2. E3 và E4 dùng cùng kiến trúc, dữ liệu, seed, thứ tự lô và ngân sách 6 epoch.
Chỉ E4 chép các vector Word2Vec vào lớp Embedding trước khi huấn luyện. Cả hai lớp Embedding đều được cập nhật.
Viết giả thuyết có ngày trước khi chạy E4. So sánh hai mô hình bằng macro-F1 và F1 từng nhãn trên validation.

Dữ liệu: **Vietnamese AI Feedback OOD 3125**, giấy phép CC BY 4.0. Dùng các tệp CSV đã chia sẵn từ gói lớp. Xem README để đặt tệp và chạy. Tuần 1–2 chỉ đọc train và validation. Nhận tệp test của lớp vào tuần 3.

In [1]:
import hashlib
import io
import json
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

LABELS = ["negative", "neutral", "positive"]
DATA_VERSION = "ood3125-de7a66b-group42-v1"
CHECKSUMS = {'train': '17e2029dcf20ba77ef95b62b77e4a0458eabea1d36cb72e0794f790340ea1201', 'validation': 'c8148193a9d14963fbc25837f5bb6277bb907d36d1811d2cc25ea6e935847d0c', 'test': '4bee1b48b6dc277742f4d9591c602a720d4a483646e57786df40bb47f2cb7ef7'}


def text_tokens(text):
    """Tokenize word-like spans after Unicode normalization and case folding."""
    return re.findall(r"\w+", unicodedata.normalize("NFC", text).casefold())


def read_split(split):
    filename = f"ood3125_{split}.csv"
    path = Path("data") / filename
    if not path.is_file():
        path = Path(filename)
    if not path.is_file():
        raise FileNotFoundError(f"Đặt {filename} trong data/ hoặc thư mục làm việc.")
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != CHECKSUMS[split]:
        raise ValueError(f"{filename} không đúng bản chia của lớp DCT1243.")
    frame = pd.read_csv(io.BytesIO(content))
    frame = frame.rename(columns={"sentence": "text"})
    frame["label"] = frame["label"].map(dict(enumerate(LABELS)))
    return frame


def scores(gold, predicted):
    per_label = f1_score(gold, predicted, labels=LABELS, average=None, zero_division=0)
    return {"accuracy": accuracy_score(gold, predicted), "macro_f1": per_label.mean(),
            **{f"f1_{label}": value for label, value in zip(LABELS, per_label)}}

from gensim.models import Word2Vec

MODEL_SEED = 42
EMBED_DIM = 100


def fit_word2vec(tokenized_train):
    return Word2Vec(tokenized_train, vector_size=EMBED_DIM, window=5, min_count=2,
                    sg=1, negative=5, epochs=20, seed=MODEL_SEED, workers=1)

import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

MAX_TOKENS = 50
HIDDEN_DIM = 64
MAX_EPOCHS = 6
torch.set_num_threads(2)


def encode(texts, vocabulary):
    rows, lengths = [], []
    for text in texts:
        ids = [vocabulary.get(token, 1) for token in text_tokens(text)[:MAX_TOKENS]]
        lengths.append(max(1, len(ids)))
        rows.append(ids + [0] * (MAX_TOKENS - len(ids)))
    return torch.tensor(rows, dtype=torch.long), torch.tensor(lengths)


class SentimentLSTM(nn.Module):
    def __init__(self, vocabulary_size, initial_embeddings=None):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, EMBED_DIM, padding_idx=0)
        if initial_embeddings is not None:
            with torch.no_grad():
                self.embedding.weight.copy_(initial_embeddings)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(HIDDEN_DIM, len(LABELS))

    def forward(self, token_ids, lengths):
        embeddings = self.embedding(token_ids)  # [batch, time, embedding]
        packed = pack_padded_sequence(embeddings, lengths, batch_first=True,
                                      enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)  # [1, batch, hidden]
        return self.classifier(self.dropout(hidden[-1]))  # [batch, 3]


def predict_lstm(model, texts, vocabulary):
    model.eval()
    token_ids, lengths = encode(texts, vocabulary)
    batches = DataLoader(TensorDataset(token_ids, lengths), batch_size=128)
    with torch.no_grad():
        indices = torch.cat([model(ids, lens).argmax(1) for ids, lens in batches])
    return np.array(LABELS)[indices.numpy()]

from copy import deepcopy


def fit_lstm(train, validation, vocabulary, initial_embeddings):
    torch.manual_seed(MODEL_SEED)
    model = SentimentLSTM(len(vocabulary), initial_embeddings)
    train_ids, train_lengths = encode(train["text"], vocabulary)
    train_labels = torch.tensor(train["label"].map(LABELS.index).to_numpy())
    batches = DataLoader(TensorDataset(train_ids, train_lengths, train_labels),
                         batch_size=64, shuffle=True,
                         generator=torch.Generator().manual_seed(MODEL_SEED))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for ids, lengths, gold in batches:
            optimizer.zero_grad()
            logits = model(ids, lengths)
            loss = loss_fn(logits, gold)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(gold)
        predicted = predict_lstm(model, validation["text"], vocabulary)
        value = scores(validation["label"], predicted)["macro_f1"]
        history.append({"epoch": epoch, "train_loss": total_loss / len(train),
                        "macro_f1": value})
        if epoch == 1 or value > best_f1:
            best_f1, best_epoch = value, epoch
            best_state = deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch

def checkpoint_digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

In [2]:
STUDENT_ID = "3124410066"
if not STUDENT_ID.isascii() or not STUDENT_ID.isdigit() or len(STUDENT_ID) < 3:
    raise ValueError("Điền MSSV gồm ít nhất ba chữ số rồi chạy lại ô này.")
PERSONAL_SEED = int(STUDENT_ID[-3:])
print("MSSV:", STUDENT_ID, "| Seed lấy mẫu:", PERSONAL_SEED)

MSSV: 3124410066 | Seed lấy mẫu: 66


In [3]:
train, validation = read_split("train"), read_split("validation")
word2vec = fit_word2vec([text_tokens(text) for text in train["text"]])
vocabulary = {"<pad>": 0, "<unk>": 1}
vocabulary.update({token: i for i, token in enumerate(word2vec.wv.index_to_key, 2)})
torch.manual_seed(MODEL_SEED)
random_embeddings = SentimentLSTM(len(vocabulary)).embedding.weight.detach().clone()
print("Vocab / embedding / hidden:", len(vocabulary), EMBED_DIM, HIDDEN_DIM)

Vocab / embedding / hidden: 1216 100 64


## C3 · LSTM và giả thuyết
Đọc `forward` và `fit_lstm`. Trong báo cáo, ghi kích thước tensor ở ba vị trí
(Embedding, trạng thái ẩn cuối, đầu ra) và giải thích vai trò của độ dài thật trong xử lý padding.
Chạy E3. Ghi giả thuyết và ngày viết tại ô văn bản bên dưới trước khi chạy E4.

In [4]:
e3, history_e3, epoch_e3 = fit_lstm(train, validation, vocabulary, None)

In [8]:
display(history_e3)
print("Epoch tốt nhất của E3:", epoch_e3)

,epoch,train_loss,macro_f1
0,1,1.024738,0.633250
1,2,0.742537,0.745258
2,3,0.268657,0.907856
3,4,0.104634,0.926460
4,5,0.055499,0.952409
5,6,0.040329,0.952434


Epoch tốt nhất của E3: 6


**Giả thuyết và ngày viết** — điền cơ chế, dự đoán đo được và điều kiện không ủng hộ dự đoán.

> ⚠️ Lưu ý trung thực: mục này được viết **sau** khi E3 và E4 đã chạy xong (commit 44d1b6e lúc 02:37 ngày 24/11/2026 đã có kết quả E4), không có commit riêng trước khi chạy E4. Phần dưới là lập luận cơ chế, không phải dự đoán đăng ký trước.
- Ngày viết: 24/11/2026 (sau khi chạy E4)
- Cơ chế (vì sao dự đoán như vậy): Word2Vec Skip-gram đã học trên chính 2500 câu train, nên các token xuất hiện trong ngữ cảnh giống nhau có vector gần nhau ngay từ đầu. Khi chép vào lớp Embedding, LSTM không phải học biểu diễn từ con số 0 mà bắt đầu từ một không gian đã có cấu trúc, nên kỳ vọng mô hình học nhanh hơn ở các epoch đầu. Tuy nhiên ở tuần 1 đã thấy vector Word2Vec không mã hóa cực tính cảm xúc tốt (cosine "tốt"–"hay" chỉ 0,2171), và cả hai lớp Embedding đều được cập nhật khi huấn luyện, nên lợi thế có thể chủ yếu nằm ở tốc độ hội tụ hơn là ở điểm cuối cùng.
- Dự đoán đo được: (1) Macro-F1 validation ở epoch 1 của E4 cao hơn rõ rệt so với E3 (0,633250). (2) Macro-F1 tốt nhất của E4 cao hơn E3 (0,952434), chênh lệch dự kiến không quá lớn. (3) neutral vẫn là nhãn có F1 thấp nhất ở cả hai mô hình.
- Kết quả nào sẽ KHÔNG ủng hộ dự đoán: macro-F1 tốt nhất của E4 ≤ 0,952434; hoặc đường macro-F1 theo epoch của E4 không cao hơn E3 ở các epoch đầu; hoặc E4 cần nhiều epoch hơn E3 để đạt cùng mức macro-F1.

### Tạo ma trận khởi tạo E4
Sao chép ma trận ngẫu nhiên của E3. Trong vòng lặp, thay dòng của token `token` bằng vector Word2Vec
tương ứng. Hai dòng `<pad>` và `<unk>` dùng chung giá trị khởi tạo của E3.

**Thực hành trước khi làm mục này.** Xem [hướng dẫn Skip-gram](DCT1243_Skipgram_HuongDan.pdf) và chạy [notebook minh họa](DCT1243_Skipgram_SinhVien.ipynb) đi kèm. Nếu Colab không mở được liên kết tương đối, hãy mở hai tệp từ gói đã giải nén.

In [5]:
w2v_embeddings = random_embeddings.clone()
for token, index in vocabulary.items():
    if token in word2vec.wv:
        w2v_embeddings[index] = torch.tensor(word2vec.wv[token])

In [6]:
probe = word2vec.wv.index_to_key[0]
assert torch.allclose(w2v_embeddings[vocabulary[probe]],
                      torch.tensor(word2vec.wv[probe])), "Kiểm tra dòng Word2Vec."
assert torch.equal(w2v_embeddings[:2], random_embeddings[:2])
e4, history_e4, epoch_e4 = fit_lstm(train, validation, vocabulary, w2v_embeddings)
models = {"E3": e3, "E4": e4}
epochs = {"E3": epoch_e3, "E4": epoch_e4}
histories = {"E3": history_e3, "E4": history_e4}
validation_scores = pd.DataFrame.from_dict({
    name: scores(validation["label"], predict_lstm(model, validation["text"], vocabulary))
    for name, model in models.items()
}, orient="index")
validation_scores["epoch"] = pd.Series(epochs)
display(pd.concat(histories, names=["model"]).reset_index(level=1, drop=True))
display(validation_scores.round(4))

,epoch,train_loss,macro_f1
model,,,
E3,1,1.024738,0.633250
E3,2,0.742537,0.745258
E3,3,0.268657,0.907856
E3,4,0.104634,0.926460
E3,5,0.055499,0.952409
E3,6,0.040329,0.952434
E4,1,0.930335,0.900381
E4,2,0.282993,0.971708
E4,3,0.096417,0.968410


,accuracy,macro_f1,f1_negative,f1_neutral,f1_positive,epoch
E3,0.9519,0.9524,0.9751,0.9364,0.9458,6
E4,0.9808,0.9810,0.9901,0.9730,0.9800,5


### Khóa checkpoint
Trong từng mô hình, chọn epoch có macro-F1 validation cao nhất và lấy epoch sớm hơn nếu bằng nhau.
Chọn E3 hoặc E4 theo macro-F1 validation chưa làm tròn, ưu tiên E3 nếu bằng nhau.
Ghi một nhận xét về kết quả so với giả thuyết. Tải checkpoint và tệp quyết định xuống cùng lúc.

In [7]:
chosen = validation_scores["macro_f1"].idxmax()
checkpoint = Path(f"{STUDENT_ID}_lstm.pt")
torch.save({"vocabulary": vocabulary, "weights": models[chosen].state_dict()}, checkpoint)
decision = {"dataset": DATA_VERSION, "student_id": STUDENT_ID, "model": chosen, "epoch": epochs[chosen],
            "validation_macro_f1": float(validation_scores.loc[chosen, "macro_f1"]),
            "checkpoint_sha256": checkpoint_digest(checkpoint)}
Path(f"{STUDENT_ID}_decision.json").write_text(json.dumps(decision), encoding="utf-8")
print("QUYET_DINH", json.dumps(decision))

QUYET_DINH {"dataset": "ood3125-de7a66b-group42-v1", "student_id": "3124410066", "model": "E4", "epoch": 5, "validation_macro_f1": 0.9810239942913211, "checkpoint_sha256": "9b70a45df7a239cf495b6de8b82e493a5dff2967c225e63cfc90a251bd32be0a"}


**Nhận xét so với giả thuyết**

Vì giả thuyết được ghi sau khi đã chạy E4 và thấy kết quả (nhất là dự đoán (3) về neutral), sự trùng khớp dưới đây chỉ mang tính mô tả, không phải kiểm chứng giả thuyết. Về mặt mô tả, kết quả phù hợp với cả ba ý: epoch 1 của E4 cao hơn E3 rất xa (0,9004 so với 0,6333); macro-F1 tốt nhất của E4 cao hơn E3 khoảng 0,029; neutral thấp nhất ở cả hai. Phần "chênh lệch không quá lớn" chỉ đúng một phần: khoảng cách cuối cùng nhỏ, nhưng khoảng cách về tốc độ học rất lớn. Cần thận trọng vì: (a) E3 đạt tốt nhất ở epoch cuối và vẫn đang tăng, nên với nhiều epoch hơn E3 có thể thu hẹp khoảng cách; kết luận chỉ đúng với ngân sách 6 epoch; (b) chỉ chạy một seed (42), chưa biết độ dao động giữa các seed; (c) validation chỉ có 312 câu, chênh 0,0289 accuracy chỉ tương ứng khoảng 9 câu đoán đúng thêm. Ngoài ra, E4 (0,9810) vẫn thấp hơn baseline E1 TF-IDF + LogReg trên validation (0,9905), cần xem lại ở tập test tuần 3.